In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = 224
NUM_FRAMES = 15
BATCH_SIZE = 4
EPOCHS = 10

In [13]:
def load_video_frames(video_path):
    frames = sorted(os.listdir(video_path))[:NUM_FRAMES]
    video = []

    for frame in frames:
        img_path = os.path.join(video_path, frame)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img / 255.0
        video.append(img)

    # Padding if less frames
    while len(video) < NUM_FRAMES:
        video.append(np.zeros((IMG_SIZE, IMG_SIZE, 3)))

    return np.array(video, dtype=np.float32)


def generator(root_dir):
    classes = ["Celeb-real", "Celeb-synthesis"]

    for label, cls in enumerate(classes):
        class_path = os.path.join(root_dir, cls)

        for video_folder in os.listdir(class_path):
            video_path = os.path.join(class_path, video_folder)

            if os.path.isdir(video_path):
                video_tensor = load_video_frames(video_path)
                yield video_tensor, label

In [14]:
output_signature = (
    tf.TensorSpec(shape=(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.int32)
)

train_dataset = tf.data.Dataset.from_generator(
    lambda: generator("C:\\Users\\CSE\\Desktop\\Deepfake\\Deepfake-detection\\dataset_faces\\train"),
    output_signature=output_signature
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    lambda: generator("C:\\Users\\CSE\Desktop\\Deepfake\\Deepfake-detection\\dataset_faces\\val"),
    output_signature=output_signature
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_generator(
    lambda: generator("C:\\Users\\CSE\\Desktop\\Deepfake\\Deepfake-detection\\dataset_faces\\test"),
    output_signature=output_signature
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [15]:
def build_model():

    backbone = keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        pooling="avg"
    )
    backbone.trainable = False

    video_input = layers.Input(
        shape=(NUM_FRAMES, IMG_SIZE, IMG_SIZE, 3)
    )

    # Spatial features per frame
    x = layers.TimeDistributed(backbone)(video_input)

    # Temporal Transformer
    x = layers.LayerNormalization()(x)

    attention = layers.MultiHeadAttention(
        num_heads=4,
        key_dim=64
    )(x, x)

    x = layers.Add()([x, attention])
    x = layers.LayerNormalization()(x)

    ff = layers.Dense(512, activation="relu")(x)
    ff = layers.Dense(x.shape[-1])(ff)


    x = layers.Add()([x, ff])

    x = layers.GlobalAveragePooling1D()(x)

    output = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(video_input, output)

    return model


model = build_model()
model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_9       │ (None, 15, 224,   │          0 │ -                 │
│ (InputLayer)        │ 224, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_4  │ (None, 15, 1280)  │  4,049,571 │ input_layer_9[0]… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 15, 1280)  │      2,560 │ time_distributed… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 15, 1280)  │  1,312,768 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 15, 1280)  │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 15, 1280)  │      2,560 │ add_8[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 15, 512)   │    655,872 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 15, 1280)  │    656,640 │ dense_11[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 15, 1280)  │          0 │ layer_normalizat… │
│                     │                   │            │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1280)      │          0 │ add_9[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 1)         │      1,281 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,681,252 (25.49 MB)

 Trainable params: 2,631,681 (10.04 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS
)

Epoch 1/10
    625/Unknown 1291s 2s/step - accuracy: 0.9794 - loss: 0.1141

In [ ]:
for layer in model.layers:
    print(layer.name)

In [ ]:
def make_gradcam_heatmap(model, video_tensor, layer_name="top_conv"):

    # Take one frame only for GradCAM
    frame = video_tensor[0:1, 0]  # first frame

    backbone = model.layers[1].layer  # EfficientNet

    grad_model = keras.Model(
        [backbone.inputs],
        [backbone.get_layer(layer_name).output, backbone.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(frame)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)

    return heatmap.numpy()

In [ ]:
import matplotlib.pyplot as plt

sample_video, label = next(iter(test_dataset))
heatmap = make_gradcam_heatmap(model, sample_video)

plt.matshow(heatmap)
plt.colorbar()
plt.show()